# AIPS Model Training — YOLOv8 Vehicle + Plate Detector

**Before running:** Upload your dataset ZIP from Roboflow to Google Drive first.

**Runtime:** Set to GPU (Runtime → Change runtime type → T4 GPU)

**Estimated time:** 1–2 hours for 100 epochs on ~1000 images

In [ ]:
# ── Step 1: Install dependencies ──────────────────────────────────────────────
!pip install ultralytics -q
from ultralytics import YOLO
import torch
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE')
print('CUDA:', torch.cuda.is_available())

In [ ]:
# ── Step 2: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# ── Step 3: Upload and extract dataset ───────────────────────────────────────
# Change this path to wherever you saved your Roboflow dataset ZIP in Drive
DATASET_ZIP = '/content/drive/MyDrive/aips_dataset.zip'

import zipfile, os
os.makedirs('/content/dataset', exist_ok=True)

with zipfile.ZipFile(DATASET_ZIP, 'r') as z:
    z.extractall('/content/dataset')

print('Dataset extracted. Contents:')
!ls /content/dataset

In [ ]:
# ── Step 4: Check your data.yaml ─────────────────────────────────────────────
# Find the data.yaml file Roboflow generated
import glob
yamls = glob.glob('/content/dataset/**/data.yaml', recursive=True)
print('Found data.yaml at:', yamls)

if yamls:
    with open(yamls[0]) as f:
        print(f.read())
    DATA_YAML = yamls[0]
else:
    print('No data.yaml found — check your dataset ZIP structure.')

In [ ]:
# ── Step 5: Train ─────────────────────────────────────────────────────────────
# yolov8s = small model, good balance of speed and accuracy
# yolov8m = medium model, better accuracy, slower
# Start with yolov8s

model = YOLO('yolov8s.pt')   # downloads base weights automatically

results = model.train(
    data    = DATA_YAML,
    epochs  = 100,
    imgsz   = 640,
    batch   = 16,
    name    = 'aips_detector',
    device  = 0,            # GPU
    patience= 20,           # early stopping if no improvement for 20 epochs
    save    = True,
    plots   = True,
    # Data augmentation (already good defaults, but you can tune)
    hsv_h   = 0.015,        # hue augmentation
    hsv_s   = 0.7,          # saturation
    hsv_v   = 0.4,          # value/brightness
    fliplr  = 0.5,          # horizontal flip
    mosaic  = 1.0,          # mosaic augmentation
)

print('Training complete!')

In [ ]:
# ── Step 6: Evaluate the model ───────────────────────────────────────────────
model_path = 'runs/detect/aips_detector/weights/best.pt'
trained    = YOLO(model_path)
metrics    = trained.val(data=DATA_YAML)

print(f'mAP50:    {metrics.box.map50:.3f}')
print(f'mAP50-95: {metrics.box.map:.3f}')
print()
print('Target: mAP50 >= 0.80 before integrating into the pipeline.')

In [ ]:
# ── Step 7: Save model to Google Drive ───────────────────────────────────────
import shutil
SAVE_PATH = '/content/drive/MyDrive/aips_model/parkwise_detector.pt'
os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)
shutil.copy(model_path, SAVE_PATH)
print(f'Model saved to Drive: {SAVE_PATH}')
print()
print('Next: download parkwise_detector.pt and put it in aips-model/models/')

In [ ]:
# ── Step 8 (Optional): Test on a sample image ────────────────────────────────
from PIL import Image
import matplotlib.pyplot as plt

# Upload a test image using the Files panel on the left
TEST_IMAGE = '/content/test.jpg'   # change to your uploaded file

if os.path.exists(TEST_IMAGE):
    result = trained(TEST_IMAGE)[0]
    result_img = result.plot()
    plt.figure(figsize=(12, 8))
    plt.imshow(result_img[:, :, ::-1])
    plt.axis('off')
    plt.title('Detection Result')
    plt.show()
else:
    print('Upload a test image to /content/test.jpg to preview results.')